<a href="https://colab.research.google.com/github/francescopassante/GETMeshClassifier/blob/main/GET/src/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%cd /content

/content


In [1]:
%rm -rf GETMeshClassificator/
!git clone https://github.com/francescopassante/GETMeshClassificator.git
%cd /content

Cloning into 'GETMeshClassificator'...
remote: Enumerating objects: 329, done.
remote: Counting objects: 100% (329/329), done.
remote: Compressing objects: 100% (202/202), done.
remote: Total 329 (delta 101), reused 263 (delta 68), pack-reused 0 (from 0)
Receiving objects: 100% (329/329), 296.05 KiB | 10.21 MiB/s, done.
Resolving deltas: 100% (101/101), done.
/content


In [2]:
# Download dataset with gdown:
!gdown 1tkg25dLF-sgbNinNv8q6FCa-Z87d53kE
!unzip -q SHREC11_200NEIGH.zip

Downloading...
From (original): https://drive.google.com/uc?id=1tkg25dLF-sgbNinNv8q6FCa-Z87d53kE
From (redirected): https://drive.google.com/uc?id=1tkg25dLF-sgbNinNv8q6FCa-Z87d53kE&confirm=t&uuid=d059644c-1146-41a1-95e3-e164dc3916f1
To: /content/SHREC11_200NEIGH.zip
100% 2.41G/2.41G [00:36<00:00, 66.8MB/s]


In [ ]:
%cd /content
!unzip -q SHREC11_50NEIGH.zip

/content


In [3]:
%cd GETMeshClassificator/GET/src

/content/GETMeshClassificator/GET/src


In [11]:
import GET
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

train_loader, test_loader = GET.load_data(
        mesh_directory="../../../SHREC11/processed/",
        labels_file="../../../SHREC11/classes.txt",
        N=9,
        train_percent=0.7,
)
print(len(train_loader), len(test_loader))

415 178


In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = GET.GETClassifier(N=9, channels=12, heads = 2, out_classes=30).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=5e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=40, gamma=0.1)
num_params = sum(p.numel() for p in model.parameters())
print(f"Model has {num_params} parameters")

Model has 9114 parameters


In [15]:
def train(
    model,
    dataloader,
    optimizer,
    scheduler,
    criterion,
    device,
    epochs=1,
    accumulation_steps=16,
):
    model.train()
    loss_hist = []

    for epoch in range(epochs):
        total_loss = 0.0

        # Start with zeroed gradients for the first accumulation block
        optimizer.zero_grad()

        for i, mesh in enumerate(tqdm(dataloader, desc=f"Epoch {epoch + 1}/{epochs}")):
            x = mesh["x"].to(device).squeeze(0)
            neighbors = mesh["neighbors"].to(device).squeeze(0)
            mask = mesh["mask"].to(device).squeeze(0)
            parallel_transport_matrices = (
                mesh["parallel_transport_matrices"].to(device).squeeze(0)
            )
            rel_pos_u = mesh["rel_pos"].to(device).squeeze(0)
            labels = mesh["label"].to(device).long().squeeze(0)

            for name, t in {
              "x": x,
              "neighbors": neighbors,
              "mask": mask,
              "ptm": parallel_transport_matrices,
              "rel_pos": rel_pos_u,
              }.items():
                if not torch.isfinite(t).all():
                    print("Bad tensor:", name, "mesh:", mesh.get("filenumber", "unknown"))
                    raise ValueError(f"Non-finite in {name}")

            # Forward
            outputs = model(x, neighbors, mask, parallel_transport_matrices, rel_pos_u)
            if not torch.isfinite(outputs).all():
              print("Non-finite outputs on mesh:", mesh.get("filenumber", "unknown"))
              raise ValueError("Non-finite outputs")

            # Print the index of dimension with max output
            # print("mesh file: ", mesh["filenumber"])
            # print("predicted label: ", torch.argmax(outputs).item())
            # print("True label: ", labels.item())
            raw_loss = criterion(outputs.unsqueeze(0), labels.unsqueeze(0))

            if not torch.isfinite(raw_loss):
              print("Non-finite loss on mesh:", mesh.get("filenumber", "unknown"))
              raise ValueError("Non-finite loss")

            # Scale the loss for gradient accumulation
            scaled_loss = raw_loss / accumulation_steps
            scaled_loss.backward()

            # Accumulate the raw (unscaled) loss for logging
            total_loss += raw_loss.item()

            # Step and zero gradients at the accumulation boundary (or final batch)
            if (i + 1) % accumulation_steps == 0 or (i + 1) == len(dataloader):
                #torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

        # Average epoch loss (uses unscaled batch losses)
        epoch_loss = total_loss / len(dataloader)
        print("epoch_loss: ", epoch_loss)
        loss_hist.append(epoch_loss)
        scheduler.step()
        # Save checkpoint with meaningful loss value (epoch average)
        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss": epoch_loss,
        }
        save_path = f"checkpoint{epoch}.pth"
        torch.save(checkpoint, save_path)
        print(f"Saved checkpoint to {save_path} (epoch_loss={epoch_loss:.6f})")

    return loss_hist

In [16]:
train(model,
    train_loader,
    optimizer,
    scheduler,
    criterion,
    device,
    epochs=70,
    accumulation_steps=2)

Epoch 1/70:  54%|█████▎    | 223/415 [04:30<03:52,  1.21s/it]

Bad tensor: ptm mesh: tensor([417])


ValueError: Non-finite in ptm

In [19]:
mesh = next(iter(train_loader))
print(mesh.keys())

dict_keys(['x', 'neighbors', 'mask', 'parallel_transport_matrices', 'rel_pos', 'label', 'filenumber'])


In [22]:
# extract from train_loader the datum with "filenumber" = 417:
for mesh in train_loader:
    if mesh["filenumber"].item() == 417:
      broken_mesh = mesh
      break

In [26]:
for i,ptm in enumerate(broken_mesh["parallel_transport_matrices"]):
  print(ptm.shape)
  if not torch.isfinite(ptm).all():
    print(i)

torch.Size([591, 200, 9, 9])
0
